## Data-driven EIT Reconstruction — 67-patient LOPO Cross-Validation

**Leave-One-Patient-Out (LOPO-CV)** on 67 patients from 6 clinical studies
(RIBS, IRESP, ISUPPORT, FLOWTRIAL, MASHUP, REEF\_ROMA).

Dataset: ~315k frames sampled from 7.5M total.
Each patient contributes ~5,000 frames from multiple recordings.

**Excluded** (9 patients):
- PULSAR (PT73-76, 4 pts): no metadata, unknown protocol, mixed signal quality
- HIFIVE (PT56, 1 pt): no metadata, signal amplitude 2-5× below cohort
- PT30/IRESP (1 pt): signal amplitude 2-5× above IRESP cohort, clinical outlier
- PT66/MASHUP (1 pt): signal amplitude 5× below cohort (same pattern as HIFIVE)
- PT59/FLOWTRIAL (1 pt): X-Y global correlation = 0.35, poor electrode coupling
- PT06/RIBS (1 pt): X-Y global correlation = 0.28, poor electrode coupling

Two feature sets compared:
- **v1** (208 features): calibrated transimpedances `vv = FT_A·trans_A − FT_B·trans_B`
- **v1b** (416 features): raw `[trans_A | trans_B]` — bypasses Adler calibration

In [ ]:
from pathlib import Path
import csv
import gc
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

from fasteit.reconstruction.data_prep import normalize
from fasteit.reconstruction.metrics import (
    correlation_per_frame,
    error_map,
    mse_per_frame,
    summary_metrics,
)

warnings.filterwarnings('ignore')

# Dräger-inspired EIT colormap
CMAP_EIT = LinearSegmentedColormap.from_list("draeger", [
    (0.00, (0.5, 0.0, 0.5)),
    (0.50, (0.0, 0.0, 0.0)),
    (0.80, (0.0, 0.0, 1.0)),
    (1.00, (1.0, 1.0, 1.0)),
])

SAMPLE_DIR = Path("/mnt/my_data/eit/npz/sample_10k")
METADATA_CSV = Path("/mnt/my_data/eit/recording_metadata.csv")
N_REF = 50
MSE_THRESHOLD = 50
MIN_FRAMES_WARN = 0.3  # warn if patient retains < 30% of frames after filter
fs = 50.0

# Excluded patients: unknown protocol / out-of-distribution signal / poor coupling
EXCLUDE_PATIENTS = {
    56,              # HIFIVE — no metadata, signal amplitude 2-5× below cohort
    73, 74, 75, 76,  # PULSAR — no metadata, unknown protocol
    30, 68,              # IRESP outlier — signal amplitude 2-5× above IRESP cohort
    66,              # MASHUP — signal amplitude 5× below cohort (same as HIFIVE)
    59,              # FLOWTRIAL — X-Y global correlation 0.35, poor electrode coupling
    6,               # RIBS — X-Y global correlation 0.28, poor electrode coupling
}

## 1. Load 67-patient dataset from pre-built NPZ

In [ ]:
def normalize_per_recording(X, Y, rec_id, n_ref=N_REF):
    """Divisive normalisation per recording segment.

    Removes baseline offset and scales by reference: x_norm = (x - ref) / |ref|.
    Channels where |ref| < 1.0 are clamped to 1.0 to avoid numerical explosion,
    gracefully degrading to subtractive normalisation for near-zero baselines.
    """
    X_out = np.empty_like(X)
    Y_out = np.empty_like(Y)
    for rid in np.unique(rec_id):
        idx = np.where(rec_id == rid)[0]
        x_seg, y_seg = X[idx], Y[idx]
        n = min(n_ref, len(x_seg))
        X_out[idx] = x_seg - x_seg[:n].mean(axis=0)
        Y_out[idx] = y_seg - y_seg[:n].mean(axis=0)
        #ref_x = x_seg[:n].mean(axis=0)
        #X_out[idx] = (x_seg - ref_x) / np.clip(np.abs(ref_x), 1.0, None)
        #ref_y = y_seg[:n].mean(axis=0)
        #Y_out[idx] = (y_seg - ref_y) / np.clip(np.abs(ref_y), 1.0, None)
    return X_out, Y_out

# Load metadata for study/sex/mode annotations
meta_by_patient = {}
with open(METADATA_CSV) as f:
    for row in csv.DictReader(f):
        pid = int(row["patient_id"])
        if pid not in meta_by_patient:
            meta_by_patient[pid] = {
                "study": row["study"],
                "sex": row["sex"],
                "ventilation_mode": row["ventilation_mode"],
            }

# Load all patients
npz_files = sorted(SAMPLE_DIR.glob("patient_*.npz"))
print(f"Loading from {SAMPLE_DIR} (excluding {len(EXCLUDE_PATIENTS)} patients: {sorted(EXCLUDE_PATIENTS)}) ...")

X_vv_list, X_raw_list, Y_list = [], [], []
patient_ids = []
total_removed = 0
n_excluded = 0
retention_log = []  # track per-patient frame retention

for npz_path in npz_files:
    data = np.load(npz_path)
    pid = int(data["patient_id"])

    if pid in EXCLUDE_PATIENTS:
        n_excluded += 1
        continue

    rec_id = data["rec_id"]
    n_total = data["X_vv"].shape[0]

    # Divisive normalisation per-recording
    xvv_n, y_n = normalize_per_recording(data["X_vv"], data["Y"], rec_id)
    xraw_n, _ = normalize_per_recording(data["X_raw"], data["Y"], rec_id)

    # Filter corrupted frames (MSE on Y after normalisation)
    frame_mse = np.mean(y_n ** 2, axis=1)
    good = frame_mse < MSE_THRESHOLD
    n_bad = (~good).sum()
    total_removed += n_bad

    xvv_n, xraw_n, y_n = xvv_n[good], xraw_n[good], y_n[good]
    retention = good.sum() / n_total
    retention_log.append((pid, n_total, good.sum(), retention))

    if n_bad > 0:
        warn = " ⚠️ LOW RETENTION" if retention < MIN_FRAMES_WARN else ""
        print(f"  PT{pid:02d}: {n_bad}/{n_total} removed ({retention:.0%} kept){warn}")

    X_vv_list.append(xvv_n)
    X_raw_list.append(xraw_n)
    Y_list.append(y_n)
    patient_ids.append(pid)

n_patients = len(patient_ids)
total_frames = sum(x.shape[0] for x in X_vv_list)
mem_gb = sum(x.nbytes for x in X_vv_list + X_raw_list + Y_list) / 1e9

# Retention summary
retentions = [r[3] for r in retention_log]
low_retention = [r for r in retention_log if r[3] < MIN_FRAMES_WARN]

print(f"\n  {n_patients} patients loaded ({n_excluded} excluded), {total_frames:,} frames ({total_removed} removed)")
print(f"  Retention: mean={np.mean(retentions):.0%}, min={np.min(retentions):.0%}")
if low_retention:
    print(f"  ⚠️  {len(low_retention)} patients with <{MIN_FRAMES_WARN:.0%} retention:")
    for pid, nt, ng, ret in low_retention:
        print(f"      PT{pid:02d}: {ng}/{nt} frames ({ret:.0%})")
print(f"  Memory: {mem_gb:.1f} GB")
print(f"  Studies: {sorted(set(meta_by_patient[p]['study'] for p in patient_ids))}")

## 2. Alpha selection via GCV

Select regularisation strength on pooled data (all 67 patients).
RidgeCV with Generalised Cross-Validation — efficient, no data leakage.

In [ ]:
alpha_grid = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]

# Pool a stratified subset for GCV (first 2000 frames per patient)
#X_gcv_vv = np.concatenate([x[:2000] for x in X_vv_list])
#X_gcv_raw = np.concatenate([x[:2000] for x in X_raw_list])
#Y_gcv = np.concatenate([y[:2000] for y in Y_list])

#sc_vv = StandardScaler().fit(X_gcv_vv)
#sc_raw = StandardScaler().fit(X_gcv_raw)
#sc_y = StandardScaler().fit(Y_gcv)
#Y_gcv_scaled = sc_y.transform(Y_gcv)

#rcv_v1 = RidgeCV(alphas=alpha_grid).fit(sc_vv.transform(X_gcv_vv), Y_gcv_scaled)
#rcv_v1b = RidgeCV(alphas=alpha_grid).fit(sc_raw.transform(X_gcv_raw), Y_gcv_scaled)

#ALPHA_V1 = float(rcv_v1.alpha_)
ALPHA_V1 = 0.1
#ALPHA_V1B = float(rcv_v1b.alpha_)
ALPHA_V1B = 0.1

#del X_gcv_vv, X_gcv_raw, Y_gcv, Y_gcv_scaled, sc_y, rcv_v1, rcv_v1b
#gc.collect()

print(f"GCV selected alpha_v1  = {ALPHA_V1:g}")
print(f"GCV selected alpha_v1b = {ALPHA_V1B:g}")

## 3. LOPO Cross-Validation

For each of the 67 patients:
1. Train on 66 patients
2. Test on the held-out patient (~5k frames)
3. Compute all metrics

Both v1 (208 features) and v1b (416 features) evaluated per fold.

In [ ]:
results_v1 = []
results_v1b = []
all_corr_v1b = []  # spatial correlations for histogram
example_fold = None  # save one fold for detailed visualisation

t0 = time.time()

for test_idx in range(n_patients):
    test_pid = patient_ids[test_idx]
    train_idx = [i for i in range(n_patients) if i != test_idx]

    Y_tr = np.concatenate([Y_list[i] for i in train_idx])
    Y_te = Y_list[test_idx]

    # Scale Y on training data, apply to both
    scaler_y = StandardScaler().fit(Y_tr)
    Y_tr_sc = scaler_y.transform(Y_tr)

    # ---- v1 (208 features) ----
    X_tr = np.concatenate([X_vv_list[i] for i in train_idx])
    scaler_x = StandardScaler().fit(X_tr)
    X_tr = scaler_x.transform(X_tr)

    model = Ridge(alpha=ALPHA_V1).fit(X_tr, Y_tr_sc)
    Y_pred_v1 = scaler_y.inverse_transform(
        model.predict(scaler_x.transform(X_vv_list[test_idx]))
    )

    m = summary_metrics(Y_te, Y_pred_v1)
    m["patient_id"] = test_pid
    m["study"] = meta_by_patient[test_pid]["study"]
    results_v1.append(m)
    del X_tr, model

    # ---- v1b (416 features) ----
    X_tr = np.concatenate([X_raw_list[i] for i in train_idx])
    scaler_x = StandardScaler().fit(X_tr)
    X_tr = scaler_x.transform(X_tr)

    model = Ridge(alpha=ALPHA_V1B).fit(X_tr, Y_tr_sc)
    Y_pred_v1b = scaler_y.inverse_transform(
        model.predict(scaler_x.transform(X_raw_list[test_idx]))
    )

    m = summary_metrics(Y_te, Y_pred_v1b)
    m["patient_id"] = test_pid
    m["study"] = meta_by_patient[test_pid]["study"]
    results_v1b.append(m)

    # Collect spatial correlations for histogram
    all_corr_v1b.append(correlation_per_frame(Y_te, Y_pred_v1b))

    # Save one example fold for visualisation
    if example_fold is None:
        example_fold = {
            "Y_true": Y_te.copy(),
            "Y_pred_v1": Y_pred_v1.copy(),
            "Y_pred_v1b": Y_pred_v1b.copy(),
            "patient_id": test_pid,
        }

    del X_tr, Y_tr, Y_tr_sc, model, Y_pred_v1, Y_pred_v1b
    gc.collect()

    if (test_idx + 1) % 10 == 0 or test_idx == 0:
        elapsed = time.time() - t0
        eta = elapsed / (test_idx + 1) * (n_patients - test_idx - 1)
        print(f"  Fold {test_idx+1:2d}/{n_patients} "
              f"(PT{test_pid:02d} {meta_by_patient[test_pid]['study']}) "
              f"R²={results_v1b[-1]['r2']:.3f}  "
              f"[{elapsed:.0f}s elapsed, ~{eta:.0f}s remaining]")

elapsed = time.time() - t0
print(f"\nLOPO-CV complete: {n_patients} folds in {elapsed:.0f}s")

## 4. Aggregate metrics — v1 vs v1b

In [ ]:
def agg(results, key):
    vals = [r[key] for r in results]
    return np.mean(vals), np.std(vals), np.median(vals)

print(f"{'Metric':<25s} {'v1 (208)':>18s} {'v1b (416)':>18s}")
print("-" * 63)

for key in ["r2", "rmse", "mse_mean", "corr_spatial_mean", "corr_global"]:
    m1, s1, _ = agg(results_v1, key)
    m2, s2, _ = agg(results_v1b, key)
    print(f"{key:<25s} {m1:>8.4f} ± {s1:<7.4f} {m2:>8.4f} ± {s2:<7.4f}")

print()
# Win rate: how often v1b beats v1 per patient
wins = sum(1 for a, b in zip(results_v1, results_v1b) if b["r2"] > a["r2"])
print(f"v1b wins on R²: {wins}/{n_patients} patients ({100*wins/n_patients:.0f}%)")

## 5. Per-study breakdown

How does the model generalise across different clinical studies?
Each study has different patient populations, ventilation modes, and protocols.

In [ ]:
studies = sorted(set(r["study"] for r in results_v1b))

print(f"{'Study':<15s} {'N_pt':>5s} {'R²':>12s} {'Spatial corr':>14s} {'Global corr':>14s}")
print("-" * 65)

for study in studies:
    recs = [r for r in results_v1b if r["study"] == study]
    r2s = [r["r2"] for r in recs]
    sc = [r["corr_spatial_mean"] for r in recs]
    gc_ = [r["corr_global"] for r in recs]
    print(f"{study:<15s} {len(recs):>5d} "
          f"{np.mean(r2s):>6.3f}±{np.std(r2s):<5.3f} "
          f"{np.mean(sc):>7.3f}±{np.std(sc):<5.3f} "
          f"{np.mean(gc_):>7.3f}±{np.std(gc_):<5.3f}")

## 6. Per-patient R² — v1b scatter plot by study

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), layout="constrained")

study_colors = {s: f"C{i}" for i, s in enumerate(studies)}
for study in studies:
    recs = [r for r in results_v1b if r["study"] == study]
    pids = [r["patient_id"] for r in recs]
    r2s = [r["r2"] for r in recs]
    ax.scatter(pids, r2s, c=study_colors[study], label=study, s=50, zorder=3)

ax.axhline(np.mean([r["r2"] for r in results_v1b]), ls="--", c="grey", lw=0.8, label="mean")
ax.set_xlabel("Patient ID")
ax.set_ylabel("R² (LOPO fold)")
ax.set_title("Per-patient R² — v1b (416 features) — LOPO-CV")
ax.legend(ncol=4, fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

## 7. Global signal overlay (example fold)

The global EIT signal (sum of all pixels) is proportional to tidal volume.
Shown here for the first LOPO fold (held-out patient).

In [ ]:
ef = example_fold
g_true = ef["Y_true"].sum(axis=1)
g_v1 = ef["Y_pred_v1"].sum(axis=1)
g_v1b = ef["Y_pred_v1b"].sum(axis=1)

n_show = min(500, len(g_true))
t = np.arange(n_show) / fs

fig, axes = plt.subplots(2, 1, figsize=(14, 5), layout="constrained", sharex=True)

axes[0].plot(t, g_true[:n_show], lw=0.9, label=".bin (ground truth)", color="C1")
axes[0].plot(t, g_v1[:n_show], lw=0.9, label="v1 (208)", color="C0", alpha=0.8)
axes[0].set_ylabel("Global signal")
axes[0].set_title(f"LOPO fold: PT{ef['patient_id']:02d} held out")
axes[0].legend(fontsize=8)

axes[1].plot(t, g_true[:n_show], lw=0.9, label=".bin (ground truth)", color="C1")
axes[1].plot(t, g_v1b[:n_show], lw=0.9, label="v1b (416)", color="C2", alpha=0.8)
axes[1].set_ylabel("Global signal")
axes[1].set_xlabel("Time (s)")
axes[1].legend(fontsize=8)
plt.show()

## 8. Image comparison — inspiratory peak (example fold)

In [ ]:
peak_idx = int(np.argmax(g_true[:200]))

img_true = ef["Y_true"][peak_idx].reshape(32, 32)
img_v1 = ef["Y_pred_v1"][peak_idx].reshape(32, 32)
img_v1b = ef["Y_pred_v1b"][peak_idx].reshape(32, 32)

vmax = max(abs(img_true.min()), abs(img_true.max()))

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
for ax, img, title in zip(axes, [img_true, img_v1, img_v1b],
                           [".bin ground truth", "v1 (208)", "v1b (416)"]):
    im = ax.imshow(img, cmap=CMAP_EIT, vmin=-vmax, vmax=vmax, origin="lower")
    ax.set_title(title)
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="ΔZ (a.u.)")
fig.suptitle(f"PT{ef['patient_id']:02d} — frame {peak_idx} (inspiratory peak)", fontsize=12)
plt.show()

## 9. Spatial correlation distribution (all folds)

Spatial correlations collected during the LOPO loop — pooled across all
76 folds into a single histogram.

In [ ]:
all_corr = np.concatenate(all_corr_v1b)

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.hist(all_corr, bins=100, color="C2", alpha=0.7, edgecolor="white", lw=0.3)
ax.axvline(np.median(all_corr), ls="--", color="k", lw=1,
           label=f"median = {np.median(all_corr):.3f}")
ax.axvline(np.percentile(all_corr, 5), ls=":", color="grey", lw=1,
           label=f"p5 = {np.percentile(all_corr, 5):.3f}")
ax.set_xlabel("Spatial correlation (Pearson r)")
ax.set_ylabel("Frame count")
ax.set_title(f"Spatial correlation — v1b LOPO-CV — {len(all_corr):,} frames")
ax.legend()
plt.show()

## Summary

### LOPO-CV on 67 patients (6 studies)

Excluded 9 patients:
- PULSAR (4 pts): no metadata, unknown protocol
- HIFIVE (1 pt): no metadata, signal amplitude 2-5× below cohort
- PT30/IRESP (1 pt): signal amplitude 2-5× above cohort, clinical outlier
- PT66/MASHUP (1 pt): signal amplitude 5× below cohort (same pattern as HIFIVE)
- PT59/FLOWTRIAL (1 pt): X-Y global correlation 0.35, poor electrode coupling
- PT06/RIBS (1 pt): X-Y global correlation 0.28, poor electrode coupling

Key findings:
- **True cross-patient generalisation** — each fold tests on a patient
  the model has never seen during training
- Per-study breakdown reveals which populations the model handles best
- Exclusion criteria are justified by missing metadata, out-of-distribution signal,
  or poor X-Y coupling — not by post-hoc metric selection